# 03 — Build a GPT

**Companion to [Chapter 05](../05-transformer-architecture.md).**

Assemble a complete, architecturally-correct GPT-2 from parts:
LayerNorm, GELU, FeedForward, TransformerBlock, GPTModel — then count its
parameters and generate text with it.

The weights are random, so the output is gibberish. That is the correct result:
the *architecture* is right, the *knowledge* comes from training (notebook 04).

## 1. Layer normalization

Normalizes each token's feature vector independently — not across the batch. That is why it suits variable-length sequences.

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(123)

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5                                    # prevents divide-by-zero
        self.scale = nn.Parameter(torch.ones(emb_dim))     # learned
        self.shift = nn.Parameter(torch.zeros(emb_dim))    # learned

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.scale * (x - mean) / torch.sqrt(var + self.eps) + self.shift


x = torch.randn(2, 4, 16) * 5 + 3
out = LayerNorm(16)(x)

print(f"before: mean {x.mean(-1)[0,0]:+.3f}  std {x.std(-1)[0,0]:.3f}")
print(f"after : mean {out.mean(-1)[0,0]:+.3f}  std {out.std(-1)[0,0]:.3f}")
print("\nNote the normalization is over the LAST dim (features of one token),")
print("not over the batch.")

before: mean +3.431  std 4.009
after : mean +0.000  std 1.033

Note the normalization is over the LAST dim (features of one token),
not over the batch.


## 2. GELU

A smooth ReLU. The small leak for negative values gives a continuous gradient everywhere, which trains better.

In [2]:
class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * x ** 3)))


print(f"{'x':>6} {'ReLU':>8} {'GELU':>8}")
for v in [-3.0, -1.0, -0.5, 0.0, 0.5, 1.0, 3.0]:
    t = torch.tensor([v])
    print(f"{v:>6.1f} {torch.relu(t).item():>8.3f} {GELU()(t).item():>8.3f}")

print("\nAt x=-0.5, ReLU gives exactly 0 (dead gradient); GELU gives -0.154.")

     x     ReLU     GELU
  -3.0    0.000   -0.004
  -1.0    0.000   -0.159
  -0.5    0.000   -0.154
   0.0    0.000    0.000
   0.5    0.500    0.346
   1.0    1.000    0.841
   3.0    3.000    2.996

At x=-0.5, ReLU gives exactly 0 (dead gradient); GELU gives -0.154.


## 3. Feed-forward network

Expands 4x and back. This is where roughly two-thirds of the model's parameters live — more than attention.

In [3]:
class FeedForward(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            GELU(),
            nn.Linear(4 * emb_dim, emb_dim))

    def forward(self, x):
        return self.layers(x)


ff = FeedForward(768)
print("768 -> 3072 -> 768")
print(f"FFN parameters      : {sum(p.numel() for p in ff.parameters()):,}")
print(f"attention parameters: {4 * 768 * 768:,}  (W_q, W_k, W_v, out_proj)")
print("\nThe FFN holds about 2x what attention does.")

768 -> 3072 -> 768
FFN parameters      : 4,722,432
attention parameters: 2,359,296  (W_q, W_k, W_v, out_proj)

The FFN holds about 2x what attention does.


## 4. Why residual connections are non-negotiable

`x = x + sublayer(x)` gives gradients a direct path back to early layers.
Run this to watch gradients vanish without them.

In [4]:
class DeepNet(nn.Module):
    def __init__(self, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut
        self.layers = nn.ModuleList(
            [nn.Sequential(nn.Linear(8, 8), nn.GELU()) for _ in range(5)])

    def forward(self, x):
        for layer in self.layers:
            out = layer(x)
            x = x + out if self.use_shortcut else out
        return x


for use_shortcut in [False, True]:
    torch.manual_seed(0)
    model = DeepNet(use_shortcut)
    model(torch.ones(1, 8)).sum().backward()
    grads = [p.grad.abs().mean().item()
             for n, p in model.named_parameters() if "weight" in n]
    print(f"shortcuts={str(use_shortcut):5}  layer gradients (first -> last):")
    print("   " + "  ".join(f"{g:.5f}" for g in grads))

print("\nWithout shortcuts the first layer's gradient is far smaller.")
print("At 12 or 96 layers instead of 5, it vanishes entirely.")

shortcuts=False  layer gradients (first -> last):
   0.00380  0.00222  0.00271  0.00301  0.04970
shortcuts=True   layer gradients (first -> last):
   0.72912  0.73164  0.54144  0.54259  0.73358

Without shortcuts the first layer's gradient is far smaller.
At 12 or 96 layers instead of 5, it vanishes entirely.


## 5. The transformer block

Pre-LayerNorm: normalize *before* each sublayer, leaving the residual path clean.

In [5]:
import torch
import torch.nn as nn

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.scale * (x - mean) / torch.sqrt(var + self.eps) + self.shift


class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * x ** 3)))


class FeedForward(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim), GELU(), nn.Linear(4 * emb_dim, emb_dim))

    def forward(self, x):
        return self.layers(x)


class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout=0.0, num_heads=12, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0
        self.d_out, self.num_heads = d_out, num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(
            torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, n, _ = x.shape
        q = self.W_query(x).view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.W_key(x).view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.W_value(x).view(b, n, self.num_heads, self.head_dim).transpose(1, 2)
        scores = q @ k.transpose(2, 3) / k.shape[-1] ** 0.5
        scores = scores.masked_fill(self.mask.bool()[:n, :n], -torch.inf)
        w = self.dropout(torch.softmax(scores, dim=-1))
        ctx = (w @ v).transpose(1, 2).contiguous().view(b, n, self.d_out)
        return self.out_proj(ctx)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            cfg["emb_dim"], cfg["emb_dim"], cfg["context_length"],
            cfg["drop_rate"], cfg["n_heads"], cfg["qkv_bias"])
        self.ff = FeedForward(cfg["emb_dim"])
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        x = x + self.drop(self.att(self.norm1(x)))    # attention sublayer + residual
        x = x + self.drop(self.ff(self.norm2(x)))     # feed-forward sublayer + residual
        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop = nn.Dropout(cfg["drop_rate"])
        self.blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, idx):
        _, n = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(n, device=idx.device))
        x = self.blocks(self.drop(x))
        return self.out_head(self.final_norm(x))

print("all components defined")

all components defined


## 6. The complete model

In [6]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,     # BPE vocabulary
    "context_length": 1024,  # max sequence length
    "emb_dim": 768,          # model width
    "n_heads": 12,           # attention heads
    "n_layers": 12,          # transformer blocks
    "drop_rate": 0.1,        # dropout
    "qkv_bias": False,
}

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)

total = sum(p.numel() for p in model.parameters())
print(f"Total parameters : {total:,}")
print(f"Size at fp32     : {total * 4 / 1e6:.2f} MB")
print(f"Size at bf16     : {total * 2 / 1e6:.2f} MB")

tied = total - GPT_CONFIG_124M["vocab_size"] * GPT_CONFIG_124M["emb_dim"]
print(f"\nWith weight tying: {tied:,}  <-- the famous 'GPT-2 124M'")
print("The output head reuses the token embedding matrix, so those")
print(f"{GPT_CONFIG_124M['vocab_size'] * GPT_CONFIG_124M['emb_dim']:,} parameters are counted once, not twice.")

Total parameters : 163,009,536
Size at fp32     : 652.04 MB
Size at bf16     : 326.02 MB

With weight tying: 124,412,160  <-- the famous 'GPT-2 124M'
The output head reuses the token embedding matrix, so those
38,597,376 parameters are counted once, not twice.


### Where the parameters actually live

In [7]:
groups = {}
for name, p in model.named_parameters():
    if name.startswith("tok_emb"):    key = "token embeddings"
    elif name.startswith("pos_emb"):  key = "position embeddings"
    elif ".att." in name:             key = "attention"
    elif ".ff." in name:              key = "feed-forward"
    elif "norm" in name:              key = "layer norms"
    else:                             key = "output head"
    groups[key] = groups.get(key, 0) + p.numel()

for key, n in sorted(groups.items(), key=lambda kv: -kv[1]):
    print(f"{key:20s} {n:>12,}  {n/total*100:5.1f}%")

feed-forward           56,669,184   34.8%
token embeddings       38,597,376   23.7%
output head            38,597,376   23.7%
attention              28,320,768   17.4%
position embeddings       786,432    0.5%
layer norms                38,400    0.0%


## 7. A forward pass

In [8]:
batch = torch.randint(0, 50257, (2, 10))
print("input  (token ids):", tuple(batch.shape))

model.eval()
with torch.no_grad():
    logits = model(batch)

print("output (logits)   :", tuple(logits.shape))
print("\nOne score per vocabulary entry, at every position.")
print("When generating, only the LAST position is used.")

probs = torch.softmax(logits[0, -1], dim=-1)
top = torch.topk(probs, 5)
print("\ntop-5 predictions (random weights, so meaningless):")
for p, i in zip(top.values, top.indices):
    print(f"   token {i.item():>6}  p={p.item():.5f}")

input  (token ids): (2, 10)


output (logits)   : (2, 10, 50257)

One score per vocabulary entry, at every position.
When generating, only the LAST position is used.

top-5 predictions (random weights, so meaningless):
   token  47803  p=0.00019
   token  37655  p=0.00018
   token  34862  p=0.00017
   token  33719  p=0.00017
   token  17913  p=0.00015


## 8. Generation

Three details worth noticing in the loop below — each becomes important later:

1. `logits[:, -1, :]` — only the last position predicts the next token
2. `idx[:, -context_size:]` — beyond the context window, old tokens are dropped
3. the whole prefix is **recomputed every step** — this is what the KV cache fixes (notebook 06)

In [9]:
@torch.no_grad()
def generate_greedy(model, idx, max_new_tokens, context_size):
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        logits = model(idx_cond)[:, -1, :]
        next_id = torch.argmax(torch.softmax(logits, dim=-1), dim=-1, keepdim=True)
        idx = torch.cat([idx, next_id], dim=1)
    return idx


start = torch.tensor([[15496, 11, 314, 716]])     # "Hello, I am" in GPT-2 BPE
out = generate_greedy(model, start, max_new_tokens=6, context_size=1024)

print("input :", start.tolist()[0])
print("output:", out.tolist()[0])
print("\nRandom weights -> meaningless tokens. The architecture is correct;")
print("the knowledge comes from training. That is notebook 04.")

input : [15496, 11, 314, 716]
output: [15496, 11, 314, 716, 27018, 24086, 47843, 30961, 42348, 7267]

Random weights -> meaningless tokens. The architecture is correct;
the knowledge comes from training. That is notebook 04.


## 9. Sizing any GPT

Compute parameter counts from a config alone.

In [10]:
def gpt_params(vocab, ctx, d, n_layers, tied=True):
    tok, pos = vocab * d, ctx * d
    attn, ffn, norms = 4 * d * d, 8 * d * d, 4 * d
    total = tok + pos + n_layers * (attn + ffn + norms) + 2 * d
    return total if tied else total + vocab * d


configs = {
    "GPT-2 small":  (768, 12),
    "GPT-2 medium": (1024, 24),
    "GPT-2 large":  (1280, 36),
    "GPT-2 XL":     (1600, 48),
}
print(f"{'model':14} {'d_model':>8} {'layers':>7} {'params':>10} {'fp32':>9} {'bf16':>9}")
for name, (d, L) in configs.items():
    n = gpt_params(50257, 1024, d, L)
    print(f"{name:14} {d:>8} {L:>7} {n/1e6:>9.1f}M {n*4/1e9:>8.2f}G {n*2/1e9:>8.2f}G")

print("\nMemory rules of thumb:")
print("  inference bf16 : ~2 bytes/param   + KV cache")
print("  inference int4 : ~0.5 bytes/param + KV cache")
print("  TRAINING       : ~16 bytes/param  (weights + grads + AdamW state)")

model           d_model  layers     params      fp32      bf16
GPT-2 small         768      12     124.4M     0.50G     0.25G
GPT-2 medium       1024      24     354.6M     1.42G     0.71G
GPT-2 large        1280      36     773.6M     3.09G     1.55G
GPT-2 XL           1600      48    1556.9M     6.23G     3.11G

Memory rules of thumb:
  inference bf16 : ~2 bytes/param   + KV cache
  inference int4 : ~0.5 bytes/param + KV cache
  TRAINING       : ~16 bytes/param  (weights + grads + AdamW state)


## Exercises

1. Build GPT-2 medium by editing only the config. Confirm ~355M parameters.
2. Set `n_heads=7` with `emb_dim=768`. Read the assertion error and explain it.
3. Compute the KV cache size for `context_length=1024`: `2 x 12 layers x 768 x 1024 x 2 bytes`. Now imagine 100 users at 128k context.

**Next:** [04 — Train a Tiny LLM](04_train_a_tiny_llm.ipynb)